This script demonstrates an example of fine-tuning Phi3.5 on a QA dataset.

In [ ]:
%pip install huggingface_hub transformers peft bitsandbytes trl xformers datasets torch flash-attention

In [ ]:
# Confirm we are using a suitable runtime
!nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits

import torch

def check_gpu_memory():
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Memory: {gpu_memory:.2f} GB")
        if gpu_memory >= 40:
            print("Sufficient VRAM: At least 40 GB available")
        else:
            print("Insufficient VRAM: Less than 40 GB available")
    else:
        print("No GPU available")

check_gpu_memory()

Your runtime has 89.6 gigabytes of available RAM

You are using a high-RAM runtime!


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig, set_seed
from huggingface_hub import ModelCard, ModelCardData, HfApi
from datasets import load_dataset
from jinja2 import Template
from trl import SFTTrainer, SFTConfig
import yaml
import torch

In [ ]:
# Configurations
MODEL_ID = "microsoft/Phi-3.5-mini-instruct"
NEW_MODEL_NAME = "Phi-3.5-EGNIVIA"
DATASET_NAME = "/content/drive/My Drive/datasets/EGNIVIA-finetune-dataset"
# MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
# NEW_MODEL_NAME = "opus-samantha-phi-3-mini-4k"
# DATASET_NAME = "macadeliccc/opus_samantha"
SPLIT = "train"
MAX_SEQ_LENGTH = 2048
num_train_epochs = 1
license = "apache-2.0"
learning_rate = 1.41e-5
per_device_train_batch_size = 4
gradient_accumulation_steps = 1

if torch.cuda.is_bf16_supported():
    print("Using supported bfloat16")
    compute_dtype = torch.bfloat16
else:
    print("Using supported float16")
    compute_dtype = torch.float16

Using supported bfloat16


In [ ]:
# Load the model, tokenizer, and dataset
from google.colab import drive
drive.mount('/content/drive')

import os

# Set up the cache directory
cache_dir = "/content/drive/My Drive/huggingface_cache"
os.makedirs(cache_dir, exist_ok=True)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID,
                                                 cache_dir=cache_dir,
                                                 trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID,
                                              cache_dir=cache_dir,
                                              trust_remote_code=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# Load and split dataset
dataset = load_dataset(DATASET_NAME, split="train")
train_size = int(len(dataset) * 0.9)

train_dataset = dataset.select(range(train_size))
eval_dataset = dataset.select(range(train_size, len(dataset)))

print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Train dataset size: 360
Eval dataset size: 40


In [ ]:
# Preprocess the dataset
def formatting_prompts_func(examples):
    """
    Format examples using chat template, compatible with SFTTrainer
    """
    texts = []

    for i in range(len(examples['question'])):
        messages = [
            {"role": "system", "content": examples['context'][i]},
            {"role": "user", "content": examples['question'][i]},
            {"role": "assistant", "content": examples['answer'][i]}
        ]

        # Note: We need to create the tokenizer outside this function
        # since we don't want to load it repeatedly for each batch
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )

        texts.append(formatted)

    return texts

In [ ]:
# Check a few examples
test_samples = {k: [dataset[i][k] for i in range(3)] for k in dataset[0].keys()}
formatted = formatting_prompts_func(test_samples)
for i, text in enumerate(formatted):
    print(f"\nExample {i}:")
    print("-" * 50)
    print(text)


Example 0:
--------------------------------------------------
<|system|>
EGNIVIA pioneered accelerated computing to help solve the most challenging computational problems, leveraging its GPU architecture to create platforms for scientific computing, AI, and more.<|end|>
<|user|>
What technology does EGNIVIA utilize for accelerated computing to address computational challenges?<|end|>
<|assistant|>
EGNIVIA utilizes GPU architecture for accelerated computing, addressing complex computational problems across various fields.<|end|>
<|endoftext|>

Example 1:
--------------------------------------------------
<|system|>
Its parallel processing capabilities, supported by thousands of computing cores, are essential to running deep learning algorithms, which allow computers and robots to simulate human intelligence.<|end|>
<|user|>
What role do EGNIVIA's GPUs play in the field of artificial intelligence?<|end|>
<|assistant|>
EGNIVIA's GPUs are essential for running deep learning algorithms, en

In [ ]:
# Define the training arguments
args = TrainingArguments(
    eval_strategy="steps",
    per_device_train_batch_size=7,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=1e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_steps=-1,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_steps=10,
    output_dir=NEW_MODEL_NAME,
    optim="paged_adamw_32bit",
    lr_scheduler_type="linear",
)

# Create SFTConfig
sft_config = SFTConfig(
    output_dir=NEW_MODEL_NAME,
    dataset_text_field="text",
    max_seq_length=128,
)

In [ ]:
# Start the fine-tuning process
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_prompts_func,
    output_dir="/content/Phi-3.5-EGNIVIA/"
)
trainer.train()

tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Step,Training Loss,Validation Loss
10,5.223300,0.486703
20,1.496500,0.223258
30,0.637000,0.136831


TrainOutput(global_step=39, training_loss=1.9811240220681214, metrics={'train_runtime': 862.9667, 'train_samples_per_second': 1.251, 'train_steps_per_second': 0.045, 'total_flos': 3933611944335360.0, 'train_loss': 1.9811240220681214, 'epoch': 3.0})

In [ ]:
# Export the training data log
from google.colab import files

!zip -r wandb.zip /content/wandb/
files.download("wandb.zip")

In [ ]:
# Re-load the new model and run inference
torch.cuda.empty_cache()
set_seed(2024)
# Replace this with the Hugging Face repository name
model_name = "/content/Phi-3.5-EGNIVIA/"
model = AutoModelForCausalLM.from_pretrained(model_name,
                                                 trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_name,
                                              trust_remote_code=True)

